In [129]:
"""GPU VMC for spinful Fermi-Hubbard with bMPS reuse.

Uses fPEPS_Model_reuse_GPU with cached boundary MPS environments
for incremental updates during sampling and energy evaluation.

Run:
    torchrun --nproc_per_node=<N> vmc_run_fpeps_reuse.py
    torchrun --nproc_per_node=1 vmc_run_fpeps_reuse.py
"""
import json
import os

import torch
import torch.distributed as dist

from vmc_torch.GPU.VMC import (
    VMC_GPU,
    VMCLoopConfig,
    VMCWarmupConfig,
    print_sampling_settings,
    setup_distributed,
)
from vmc_torch.GPU.hamiltonian import (
    spinful_Fermi_Hubbard_square_lattice_torch,
)
from vmc_torch.GPU.models import (
    fPEPS_Model_GPU,
)
from vmc_torch.GPU.optimizer import (
    DecayScheduler,
    DistributedSRMinresGPU,
    DistributedMinSRGPU,
    MinSRGPU,
    SGDGPU,
)
from vmc_torch.GPU.sampler import (
    MetropolisExchangeSpinfulSamplerGPU,
)
from vmc_torch.GPU.vmc_setup import (
    initialize_walkers,
    setup_linalg_hooks,
)
from vmc_torch.GPU.vmc_utils import (
    evaluate_energy,
    random_initial_config,
)
from vmc_torch.GPU.run_scripts.vmcconfig import VMCConfig
import os
import pickle

import autoray as ar
import quimb.tensor as qtn
import symmray as sr
dtype = torch.float64
DEFAULT_DATA_ROOT = (
    '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/GPU/data'
)

# import autoray as ar
# from symmray.linalg import (
#     svd_rand_truncated,
#     svd_truncated
# )
# # ar.register_function("symmray", "svd_truncated", svd_truncated)
# ar.register_function("symmray", "svd_truncated", svd_rand_truncated)

vmc_cfg = VMCConfig(
    batch_size=512,
    ns_per_rank=512,
    grad_batch_size=128,
    vmc_steps=1000,
    burn_in_steps=5,
    learning_rate=0.1,
    sr_diag_shift=5e-4,
    use_distributed_sr_minres=True,
    sr_rtol=1e-4,
    offload_grad_to_cpu=True,
    use_log_amp=True,
    use_export_compile=False,
    save_every=10,
    resume_step=0,
    verbose=False,
)
vmc_cfg.lr_scheduler = DecayScheduler(
    init_lr=vmc_cfg.learning_rate,
    decay_rate=0.9, patience=50,
)
warmup_cfg = VMCWarmupConfig(
    use_export_compile=vmc_cfg.use_export_compile,
    grad_batch_size=vmc_cfg.grad_batch_size,
    use_log_amp=vmc_cfg.use_log_amp,
    run_sampling=False,
    run_locE=False,
    run_grad=False,
)

def load_or_generate_peps(
    Lx, Ly, t, U, N_f, D, seed=42, dtype=torch.float64, scale_factor=4,
    data_root=DEFAULT_DATA_ROOT, file_path=None, 
):
    """Load a pre-trained fPEPS from disk, or generate a random one."""
    try:
        u1z2 = True
        appendix = '_U1SU' if u1z2 else ''
        if file_path is not None:
            base = file_path
        else:
            base = (
                f"{data_root}/{Lx}x{Ly}/t={t}_U={U}"
                f"/N={N_f}/Z2/D={D}/"
            )
        params_path = base + f"peps_su_params{appendix}.pkl"
        skeleton_path = base + f"peps_skeleton{appendix}.pkl"

        with open(params_path, 'rb') as f:
            params_pkl = pickle.load(f)
        with open(skeleton_path, 'rb') as f:
            skeleton = pickle.load(f)

        peps = qtn.unpack(params_pkl, skeleton)

        for ts in peps.tensors:
            ts.modify(data=ts.data.to_flat() * scale_factor)
        for site in peps.sites:
            peps[site].data._label = site
            peps[site].data.indices[-1]._linearmap = (
                (0, 0), (1, 0), (1, 1), (0, 1)
            )
    except Exception as e:
        import symmray as sr

        print(
            f'Could not load PEPS from pickle: {e}. '
            f'Generating random PEPS instead.'
        )
        peps = sr.networks.PEPS_fermionic_rand(
            "Z2",
            Lx,
            Ly,
            D,
            phys_dim=4,
            subsizes="equal",
            flat=True,
            seed=seed,
            dtype=str(dtype).split(".")[-1],
        )
    return peps

setup_linalg_hooks(
    jitter=1e-8, qr_via_eigh=False,
    cholesky_qr=True, cholesky_qr_adaptive_jitter=False,
    nonuniform_diag=True,
)
torch.set_default_dtype(dtype)

# rank, world_size, device = setup_distributed()
# device = torch.device(f"cuda:1")
# torch.set_default_device(device)
device = torch.device("cuda:0")
# torch.manual_seed(42 + rank)

# ========== System parameters ==========
Lx, Ly = 3, 2
N_sites = Lx * Ly
t = 1.0
U = 8.0
N_f = N_sites - 2
n_fermions_per_spin = (N_f // 2, N_f // 2)
D = 4  # PEPS bond dimension (start small)
chi = -1  # boundary bond dim

# ========== Hamiltonian ==========
H = spinful_Fermi_Hubbard_square_lattice_torch(
    Lx,
    Ly,
    t,
    U,
    N_f,
    pbc=False,
    n_fermions_per_spin=n_fermions_per_spin,
    no_u1_symmetry=False,
    gpu=True,
)
H.precompute_hops_gpu(device)
graph = H.graph

# ========== Variational state (fPEPS reuse model) ==========
fpeps_base = (
    f"{DEFAULT_DATA_ROOT}/{Lx}x{Ly}/t={t}_U={U}"
    f"/N={N_f}/Z2/D={D}/"
)
peps = load_or_generate_peps(
    Lx, Ly, t, U, N_f, D,
    seed=42, dtype=dtype,
    file_path=fpeps_base,
    scale_factor=4,
)
# peps = sr.networks.PEPS_fermionic_rand(
#     "Z2",
#     Lx,
#     Ly,
#     D,
#     phys_dim=4,
#     # put an odd number of odd sites in, for testing
#     site_charge=lambda site: int(site in [(0, 0), (0, 1), (1, 0)]),
#     subsizes="equal",
#     flat=True,
#     seed=42,
# )
# peps.apply_to_arrays(lambda x: torch.tensor(x, device=device, dtype=dtype))

model = fPEPS_Model_GPU(
    tn=peps.copy(),
    max_bond=chi,
    dtype=dtype,
    contract_boundary_opts={
        'mode': 'mps',
        'equalize_norms': 1.0,
        'canonize': True,
        'compress_opts': {
            'seed': 42
        }
    },
)
model.to(device)

fPEPS_Model_GPU(
  (params): ParameterList(
      (0): Parameter containing: [torch.float64 of size 4x2x2x2 (cuda:0)]
      (1): Parameter containing: [torch.float64 of size 4x2x2x2 (cuda:0)]
      (2): Parameter containing: [torch.float64 of size 8x2x2x2x2 (cuda:0)]
      (3): Parameter containing: [torch.float64 of size 8x2x2x2x2 (cuda:0)]
      (4): Parameter containing: [torch.float64 of size 4x2x2x2 (cuda:0)]
      (5): Parameter containing: [torch.float64 of size 4x2x2x2 (cuda:0)]
  )
)

In [130]:
edges = qtn.edges_2d_square(Lx, Ly)
sites = [(i, j) for i in range(Lx) for j in range(Ly)]
terms = sr.hamiltonians.ham_fermi_hubbard_from_edges(
    "Z2",
    edges=edges,
    U=8,
    mu=0,
)
terms = {k: v.to_flat() for k, v in terms.items()}
eref = peps.compute_local_expectation_exact(terms, normalized=True)
eref

np.float64(-2.9506184817070444)

In [131]:
all_configs = H.hilbert.all_states()
all_configs1 = [c.tolist() for c in all_configs]
z2_convention_change = {0:0, 1:2, 2:3, 3:1}
for i in range(len(all_configs1)):
    all_configs1[i] = [z2_convention_change[c] for c in all_configs1[i]]
all_configs1 = torch.tensor(all_configs1, device=device, dtype=torch.long)
all_configs = torch.tensor(all_configs, device=device, dtype=torch.long)
all_configs[:2], all_configs1[:2]

(tensor([[3, 3, 0, 0, 0, 0],
         [3, 2, 1, 0, 0, 0]], device='cuda:0'),
 tensor([[1, 1, 0, 0, 0, 0],
         [1, 3, 2, 0, 0, 0]], device='cuda:0'))

In [132]:
psi_vec = model(all_configs)
H_dense = torch.tensor(H.to_dense(), device=device, dtype=dtype)
E = psi_vec.conj() @ (H_dense @ psi_vec) / (psi_vec.conj() @ psi_vec)
E

tensor(-2.6083, device='cuda:0', grad_fn=<DivBackward0>)

In [133]:
def get_amp(peps, x):
    amp = peps.isel({
        peps.site_ind(site): x[i]
        for i, site in enumerate(peps.sites)
    })
    return amp.contract()
psi_vec_benchmark = torch.tensor(
    [get_amp(peps, x) for x in all_configs.cpu().tolist()],
    device=device, dtype=dtype
)
E_benchmark = psi_vec_benchmark.conj() @ (H_dense @ psi_vec_benchmark) / (psi_vec_benchmark.conj() @ psi_vec_benchmark)
E_benchmark

tensor(-2.6083, device='cuda:0')

In [134]:
peps.tensors[0].data, peps.tensors[0].data.indices[-1], peps.tensors[0].data.indices[-1]._linearmap, peps.tensors[0].data.indices[-1].linear_to_charge_and_offset(3)

(Z2FermionicArrayFlat(shape~(4, 4, 4):[+++], charge=1, num_blocks=4, dummy_modes=(0-,), label=(0, 0)),
 FlatIndex(num_charges=2, charge_size=2, dual=False),
 ((0, 0), (1, 0), (1, 1), (0, 1)),
 array([0, 1]))

In [135]:
x = all_configs[0].cpu().numpy()
print(x)
amp = peps.isel({
    peps.site_ind(site): x[i]
    for i, site in enumerate(peps.sites)
})
amp.tensors[0].data

[3 3 0 0 0 0]


Z2FermionicArrayFlat(shape~(4, 4):[++], charge=1, num_blocks=2, dummy_modes=(0-, ('squeeze', (0, 0), 2)-), label=(0, 0))

In [136]:
peps.tensors[0].data

Z2FermionicArrayFlat(shape~(4, 4, 4):[+++], charge=1, num_blocks=4, dummy_modes=(0-,), label=(0, 0))

In [137]:
import copy
peps1 = copy.deepcopy(peps)
for ts in peps1.tensors:
    ts_data = ts.data
    ts_data.indices[-1]._linearmap = None
    ts.modify(data=ts_data)

model1 = fPEPS_Model_GPU(
    tn=peps1,
    max_bond=chi,
    dtype=dtype,
    contract_boundary_opts={
        'mode': 'mps',
        'equalize_norms': 1.0,
        'canonize': True,
        'compress_opts': {
            'seed': 42
        }
    },
)
peps1.tensors[0].data, peps1.tensors[0].data.indices[-1], peps1.tensors[0].data.indices[-1]._linearmap

(Z2FermionicArrayFlat(shape~(4, 4, 4):[+++], charge=1, num_blocks=4, dummy_modes=(0-,), label=(0, 0)),
 FlatIndex(num_charges=2, charge_size=2, dual=False),
 None)

In [138]:
x1 = all_configs1[0].cpu().numpy()
print(x1)
amp1 = peps1.isel({
    peps1.site_ind(site): x1[i]
    for i, site in enumerate(peps1.sites)
})
amp1.tensors[0].data

[1 1 0 0 0 0]


Z2FermionicArrayFlat(shape~(4, 4):[++], charge=1, num_blocks=2, dummy_modes=(0-, ('squeeze', (0, 0), 2)-), label=(0, 0))

# How to use phys_dim=4 for proper compilation??

In [139]:
amp.tensors[0].data.blocks, amp1.tensors[0].data.blocks

(array([[[-2.12207822e-01, -1.26288963e-16],
         [-4.63767070e-16, -7.73245267e-16]],
 
        [[-1.94667938e-16,  3.13625072e-01],
         [-1.04766367e-15,  6.70711009e-02]]]),
 array([[[-2.12207822e-01, -1.26288963e-16],
         [-4.63767070e-16, -7.73245267e-16]],
 
        [[-1.94667938e-16,  3.13625072e-01],
         [-1.04766367e-15,  6.70711009e-02]]]))

In [140]:
amp.contract(), amp1.contract()

(np.float64(-0.0672271387474594), np.float64(-0.0672271387474594))

In [141]:
psi_vec[0], psi_vec_benchmark[0]

(tensor(-0.0672, device='cuda:0', grad_fn=<SelectBackward0>),
 tensor(-0.0672, device='cuda:0'))

In [142]:
model(all_configs[0].unsqueeze(0)), model1(all_configs1[0].unsqueeze(0))

(tensor([-0.0672], device='cuda:0', grad_fn=<MulBackward0>),
 tensor([-0.0672], grad_fn=<MulBackward0>))

In [143]:
all_configs[:2]

tensor([[3, 3, 0, 0, 0, 0],
        [3, 2, 1, 0, 0, 0]], device='cuda:0')

In [45]:
model.skeleton.tensors[0].data.indices[-1]._linearmap, model1.skeleton.tensors[0].data.indices[-1]._linearmap

(None, None)